# Module 2.16 — Boundary Conditions

Every PDE solver needs boundary conditions (BCs) to have a unique solution. This module formalises every BC type used in the course, introduces **ghost cells**, and shows how to implement each cleanly in NumPy.

**Four BC types cover virtually all of CFD:**

| Type | Form | What is specified |
|---|---|---|
| Dirichlet | $\phi = g$ | The value itself |
| Neumann | $\frac{\partial \phi}{\partial n} = g$ | The normal derivative (flux) |
| Robin | $a\phi + b\frac{\partial \phi}{\partial n} = c$ | A mix of value and flux |
| Periodic | $\phi(0) = \phi(L)$ | Wrap-around continuity |

In [17]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The Four BC Types

### Dirichlet BC

$$\phi = g \quad \text{on the boundary}$$

You prescribe the **value** of the field. Examples from our solvers:

```python
u[:, 0] = U          # inlet: u = U_inf
u[cylinder_mask] = 0 # no-slip on cylinder
p[:, -1] = 0.0       # reference pressure at outlet
```

---

### Neumann BC

$$\frac{\partial \phi}{\partial n} = g \quad \text{on the boundary}$$

$n$ is the **outward normal** direction. You prescribe the **flux**, not the value. The most common case is $g = 0$ (zero-gradient / zero-flux):

```python
u[:, -1] = u[:, -2]  # outlet: du/dx = 0
p[0, :]  = p[1, :]   # wall:   dp/dy = 0
```

---

### Robin BC

$$a\,\phi + b\,\frac{\partial \phi}{\partial n} = c$$

A **weighted mix** of value and flux. Arises naturally in convective heat transfer:

$$-k \frac{\partial T}{\partial n} = h\,(T_{\text{wall}} - T_{\infty})$$

where $k$ = thermal conductivity, $h$ = heat transfer coefficient.

---

### Periodic BC

$$\phi(x=0) = \phi(x=L), \quad \frac{\partial \phi}{\partial x}\bigg|_{x=0} = \frac{\partial \phi}{\partial x}\bigg|_{x=L}$$

The domain wraps around — what exits on the right re-enters on the left:

```python
u = np.roll(u, -1)   # advection with periodic BC
```

## 2. Physical Origin of Each BC

Every BC must come from **physics**, not numerical convenience.

| BC | Physical reason |
|---|---|
| No-slip $u=0$ at wall | Viscous fluid sticks to solid (experimental) |
| No-penetration $v=0$ at wall | Mass cannot pass through a solid |
| $\frac{\partial p}{\partial n}=0$ at wall | From wall-normal momentum: $-\frac{\partial p}{\partial n} + \nu\nabla^2 u_n = 0$; with $u_n=0$ this gives $\frac{\partial p}{\partial n}=0$ |
| $\frac{\partial u}{\partial x}=0$ at outlet | Flow is fully developed — field no longer changes in $x$ |
| Periodic | Flow repeats: infinite array of channels or cylinders |
| Dirichlet $p=0$ at outlet | Fixes the additive constant (Poisson solution is unique only up to a constant) |

## 3. Ghost Cells

### The problem with the current approach

In all our solvers, interior points are updated with the stencil:

$$u_i^{n+1} = u_i^n + \alpha \frac{\Delta t}{\Delta x^2}\left(u_{i+1}^n - 2u_i^n + u_{i-1}^n\right)$$

At boundary $i=0$, the term $u_{i-1}$ does not exist, so we must special-case boundaries:

```python
u[1:-1] = ...   # interior only
u[0]    = 0.0   # BC applied separately
```

### Ghost cells: extend by one layer on each side

```
| ghost_L | i=0 | i=1 | ... | i=N-1 | ghost_R |
```

Set ghost cell values to encode the BC. Then run the **same stencil uniformly** on all $N$ points.

### Ghost cell formulas

**Dirichlet** $\phi = g$ at the left face:

$$\text{ghost}_L = 2g - \phi_0$$

Derivation: the face between ghost and $\phi_0$ is at $x=0$. Face value = $\frac{\text{ghost}_L + \phi_0}{2} = g \implies \text{ghost}_L = 2g - \phi_0$.

**Neumann** $\frac{\partial \phi}{\partial x} = 0$ at the right face:

$$\text{ghost}_R = \phi_{N-1}$$

Derivation: $\frac{\text{ghost}_R - \phi_{N-1}}{\Delta x} = 0 \implies \text{ghost}_R = \phi_{N-1}$.

**Neumann** $\frac{\partial \phi}{\partial x} = h$ at the right face:

$$\text{ghost}_R = \phi_{N-1} + 2\,\Delta x\, h$$

**Periodic:**

$$\text{ghost}_L = \phi_{N-1}, \qquad \text{ghost}_R = \phi_0$$

## 4. Exercise — 1D Diffusion with Ghost Cells

Solve the 1D diffusion equation:

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

using ghost cells so the **same stencil runs on all $N$ points**.

**Setup:**
- Domain $x \in [0, 1]$, $N = 20$ interior nodes, $\Delta x = 1/(N+1)$
- Left BC: Dirichlet $u(0) = 1.0$
- Right BC: Neumann $\frac{\partial u}{\partial x} = 0$
- $\alpha = 0.01$, stability limit $r = \alpha \Delta t / \Delta x^2 \leq 0.5$, run 500 steps

**Expected result:** $u$ rises toward 1.0 at the left, flat profile at the right.

In [18]:
N     = 20
L     = 1.0
dx    = L / (N + 1)
alpha = 0.01
dt    = 0.4 * dx**2 / alpha   # r = 0.4 < 0.5
nt    = 500

x_int = np.linspace(dx, L - dx, N)
u = np.zeros(N)

print(f'dx = {dx:.4f}   dt = {dt:.6f}   r = {alpha*dt/dx**2:.3f}')

dx = 0.0476   dt = 0.090703   r = 0.400


### Your task

Fill in the ghost cell values below. Use the formulas from Section 3:

```python
ghost_L = 2*g - u[0]   # Dirichlet: face = g
ghost_R = u[-1]        # Neumann zero-gradient
```

Then verify:
- `u[0]` approaches $1.0$ (Dirichlet end)
- `u[-1] - u[-2]` approaches $0$ (Neumann end)

In [19]:
# Write your solution here

u[0] = 1.0 # Initial condition: u = 1 at the left boundary

g = 1.0 # Dirichlet boundary condition value at the left boundary

# Left ghost cell value (Dirichlet boundary condition) : ghost_left = 2 * g - u[0]
ghost_left = 2 * g - u[0] # g = 1.0 at the left boundary

# Right ghost cell value (Neumann boundary condition) : ghost_right = u[-1]
ghost_right = u[-1] # du/dx = 0 at the right boundary

u = np.insert(u, 0, ghost_left) # Insert left ghost cell
u = np.append(u, ghost_right) # Append right ghost cell

print(u[0])
print(u[-1] - u[-2])

1.0
0.0


## 5. Outflow (Convective) BC

Simple Neumann $\frac{\partial u}{\partial x} = 0$ works for steady flows. For unsteady flows (e.g. cylinder wake), vortices hitting the outlet can **reflect back** into the domain.

The **convective outflow BC** prevents this by advecting the solution out at free-stream speed $U$:

$$\frac{\partial u}{\partial t} + U\frac{\partial u}{\partial x} = 0$$

Discretized with explicit upwind in $x$:

$$u_{N-1}^{n+1} = u_{N-1}^n - U\frac{\Delta t}{\Delta x}\left(u_{N-1}^n - u_{N-2}^n\right)$$

```python
u[:, -1] = u[:, -1] - U * dt/dx * (u[:, -1] - u[:, -2])
```

| Situation | Recommended outlet BC |
|---|---|
| Steady flow (cavity) | Dirichlet $p = 0$ |
| Slowly varying | Neumann $\partial u/\partial x = 0$ |
| Unsteady vortices (cylinder) | Convective outflow |
| Repeating domain | Periodic |

## 6. Robin BC — Convective Heat Transfer

$$a\,\phi + b\,\frac{\partial \phi}{\partial n} = c$$

**Example:** wall with Newton's law of cooling:

$$-k\frac{\partial T}{\partial n} = h\,(T_{\text{wall}} - T_\infty)$$

Rearranged: $h\,T + k\,\frac{\partial T}{\partial n} = h\,T_\infty$ — a Robin BC with $a=h$, $b=k$, $c=h\,T_\infty$.

**Ghost cell for Robin:**

$$\text{ghost} = \phi_1 + \frac{(c - a\,\phi_0)\,2\,\Delta x}{b}$$

This will appear in the **heat exchanger project** (Module 4).

## Summary

| BC | Formula | Ghost cell | Course use |
|---|---|---|---|
| Dirichlet | $\phi = g$ | `ghost = 2g - phi[0]` | Inlet, no-slip, pressure ref |
| Neumann $g=0$ | $\partial\phi/\partial n = 0$ | `ghost = phi[0]` | Outlet, pressure wall |
| Neumann $g\neq 0$ | $\partial\phi/\partial n = g$ | `ghost = phi[0] + 2*dx*g` | Specified flux |
| Periodic | $\phi(0)=\phi(L)$ | `ghost_L=phi[-1]`, `ghost_R=phi[0]` | 1D advection |
| Robin | $a\phi+b\partial\phi/\partial n=c$ | `ghost = phi[1]+(c-a*phi[0])*2*dx/b` | Heat exchanger |
| Convective outflow | $\partial\phi/\partial t+U\partial\phi/\partial x=0$ | `phi[-1] -= U*dt/dx*(phi[-1]-phi[-2])` | Cylinder outlet |

**Rule of thumb:**
- Know the **value** at the boundary → Dirichlet
- Know the **flux** through the boundary → Neumann
- Know a **value-flux relationship** → Robin
- Domain **repeats** → Periodic
- **Unsteady vortices** leaving → Convective outflow